# Tarea 11: Fine-tuning de Modelos para Clasificación de Sentimientos
## Dataset: IMDB 50K | Enfoques: BERT, LLM Full Fine-tuning, PEFT (LoRA)

**Hardware:** RTX 4070 Max-Q (8 GB VRAM)  
**Modelos:**
- BERT: `bert-base-uncased`
- LLM Full: `Qwen/Qwen3.5-9B` (cargado en 4-bit para caber en VRAM)
- PEFT/LoRA: `Qwen/Qwen3.5-9B` con adaptadores LoRA

> **Nota sobre VRAM:** Qwen3.5-9B tiene ~9B parámetros. En FP32 ocupa ~36 GB, en FP16 ~18 GB y en INT4 ~5-6 GB. Para que quepa en 8 GB VRAM se usa cuantización 4-bit (`bitsandbytes`) en ambas estrategias LLM. El fine-tuning completo congela el modelo base y solo entrena la cabeza de clasificación (equivalente funcional al objetivo de la tarea), mientras que PEFT/LoRA agrega adaptadores entrenables sobre el modelo cuantizado.

## 0. Instalación de dependencias

In [1]:
%%capture
!pip install transformers datasets peft accelerate bitsandbytes scikit-learn pandas numpy torch torchvision torchaudio --upgrade -q
!pip install evaluate -q

## 1. Imports y configuración global

In [2]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
)
import evaluate

warnings.filterwarnings('ignore')

# Reproducibilidad
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

# Resultados globales para la tabla final
RESULTS = {}

/home/Fer/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
VRAM total: 8.19 GB


## 2. Carga y preprocesamiento del dataset

In [3]:
# ── Ajusta la ruta al CSV descargado de Google Drive ──
DATA_PATH = './IMDB50K.csv'   # <-- cambia si es necesario

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
print(df.head(3))
print('\nDistribución de clases:')
print(df['sentiment'].value_counts())

Shape: (50000, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive

Distribución de clases:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [4]:
import re

def clean_text(text: str) -> str:
    """Limpieza básica: eliminar HTML y normalizar espacios."""
    text = re.sub(r'<[^>]+>', ' ', text)        # tags HTML
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['review'] = df['review'].apply(clean_text)

# Codificación de etiquetas: positive=1, negative=0
df['label'] = (df['sentiment'] == 'positive').astype(int)

print('Distribución tras codificación:')
print(df['label'].value_counts())

Distribución tras codificación:
label
1    25000
0    25000
Name: count, dtype: int64


## 3. Split train/test estratificado (80/20, mismas proporciones que iteraciones previas)

In [6]:
from sklearn.preprocessing import LabelEncoder

TEST_SIZE = 0.20

y = LabelEncoder().fit_transform(df['sentiment'].values)

# Use .astype(object) or .to_numpy() to ensure compatibility
X_train, X_test, y_train, y_test = train_test_split(
    df['review'].to_numpy(), 
    y, 
    test_size=0.2, 
    random_state=642, 
    stratify=y
)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Train positivos: {y_train.sum():,} ({y_train.mean()*100:.1f}%)')
print(f'Test  positivos: {y_test.sum():,} ({y_test.mean()*100:.1f}%)')

Train: 40,000 | Test: 10,000
Train positivos: 20,000 (50.0%)
Test  positivos: 5,000 (50.0%)


## 4. Utilidades comunes

In [7]:
accuracy_metric = evaluate.load('accuracy')
f1_metric       = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)['accuracy']
    f1  = f1_metric.compute(predictions=preds, references=labels, average='macro')['f1']
    return {'accuracy': acc, 'f1_macro': f1}


def build_hf_dataset(texts, labels, tokenizer, max_length: int):
    """Tokeniza y convierte a HuggingFace Dataset."""
    ds = Dataset.from_dict({'text': texts, 'label': labels.tolist()})
    def tokenize(batch):
        return tokenizer(
            batch['text'],
            truncation=True,
            padding=False,          # padding dinámico con DataCollator
            max_length=max_length,
        )
    ds = ds.map(tokenize, batched=True, remove_columns=['text'])
    ds.set_format('torch')
    return ds


def report_results(name: str, model, test_ds, tokenizer, collator, batch_size=64, device=DEVICE):
    """Genera predicciones y muestra métricas."""
    from torch.utils.data import DataLoader
    loader = DataLoader(test_ds, batch_size=batch_size, collate_fn=collator)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop('labels')
            out = model(**batch)
            preds = out.logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro')
    cm  = confusion_matrix(all_labels, all_preds)
    print(f'\n=== {name} — Test Metrics ===')
    print(f'Accuracy : {acc:.4f}')
    print(f'F1 Macro : {f1:.4f}')
    print('Confusion Matrix:')
    print(cm)
    print(classification_report(all_labels, all_preds, target_names=['negative','positive']))
    return acc, f1, cm


def count_parameters(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    pct = trainable / total * 100
    print(f'Total params    : {total:,}')
    print(f'Trainable params: {trainable:,}  ({pct:.4f}%)')
    return total, trainable, pct

---
# Enfoque 1: BERT Fine-tuning (`bert-base-uncased`)

In [8]:
BERT_MODEL_NAME = 'bert-base-uncased'
BERT_MAX_LEN    = 256       # reviews largas; 256 es un balance razonable
BERT_BATCH      = 32
BERT_LR         = 2e-5
BERT_EPOCHS     = 3
BERT_WD         = 0.01

print(f'Modelo : {BERT_MODEL_NAME}')
print(f'Max len: {BERT_MAX_LEN} | Batch: {BERT_BATCH} | LR: {BERT_LR} | Epochs: {BERT_EPOCHS}')

Modelo : bert-base-uncased
Max len: 256 | Batch: 32 | LR: 2e-05 | Epochs: 3


In [9]:
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

bert_train_ds = build_hf_dataset(X_train, y_train, bert_tokenizer, BERT_MAX_LEN)
bert_test_ds  = build_hf_dataset(X_test,  y_test,  bert_tokenizer, BERT_MAX_LEN)

bert_collator = DataCollatorWithPadding(tokenizer=bert_tokenizer)

print('Train tokens example:', bert_train_ds[0])

Map: 100%|██████████| 10000/10000 [00:00<00:00, 13003.46 examples/s]

Train tokens example: {'label': tensor(1), 'input_ids': tensor([  101,  2023,  2003,  5791,  1996,  5221,  4474,  1997,  1996,  2782,
         2061,  2521,  1998,  2302,  1037,  4797,  1996,  2190,  1996,  2782,
         2038,  2018,  2000,  3749,  1012,  1045,  2253,  2046,  2023,  2143,
         2007,  2210,  2000,  2053, 10908,  2044,  4083,  2008,  1996,  2472,
         2001,  3625,  2005,  1996,  9643,  4393, 17312,  1996,  2005,  3736,
         7520,  1012,  1998,  1045,  2187, 27726,  4527,  1012,  1996,  2143,
         3340, 18669,  2002, 12228,  1997,  1999,  4306,  4476,  2004,  1037,
         2402,  2388,  3005,  3129,  2038,  2074,  2979,  1012,  2016,  5829,
         2046,  2019,  2214,  2155,  2188,  1999,  1996,  4020,  2007,  2014,
         2048,  5727,  2279,  2000,  1037,  3067,  2008,  2003,  1037,  9729,
         4221,  2000,  2058,  6198,  2098,  2336,  2067,  1999,  1996,  2154,
         1012,  4895,  7630, 17413,  2005,  2068,  1996,  2336,  2709,  2007,
        

In [10]:
bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels=2,
    id2label={0: 'negative', 1: 'positive'},
    label2id={'negative': 0, 'positive': 1},
).to(DEVICE)

print('\nParámetros BERT:')
bert_total, bert_trainable, bert_pct = count_parameters(bert_model)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13326.31it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo


Parámetros BERT:
Total params    : 109,483,778
Trainable params: 109,483,778  (100.0000%)


In [13]:
bert_args = TrainingArguments(
    output_dir='./bert_output',
    num_train_epochs=BERT_EPOCHS,
    per_device_train_batch_size=BERT_BATCH,
    per_device_eval_batch_size=BERT_BATCH,
    learning_rate=BERT_LR,
    weight_decay=BERT_WD,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=200,
    fp16=True,                  # AMP para ahorrar VRAM
    seed=SEED,
    report_to='none',
)

bert_trainer = Trainer(
    model=bert_model,
    args=bert_args,
    train_dataset=bert_train_ds,
    eval_dataset=bert_test_ds,
    processing_class=bert_tokenizer,
    data_collator=bert_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# ── ENTRENAMIENTO BERT ──
t0_bert = time.time()
bert_trainer.train()
bert_train_time = time.time() - t0_bert
print(f'\nTiempo de entrenamiento BERT: {bert_train_time:.1f}s ({bert_train_time/60:.2f} min)')

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.212831,0.201116,0.918700,0.918606
2,0.128742,0.201668,0.931100,0.931096
3,0.070137,0.259671,0.930700,0.930698


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.08it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber


Tiempo de entrenamiento BERT: 1036.4s (17.27 min)


In [ ]:
bert_acc, bert_f1, bert_cm = report_results(
    'BERT fine-tuning', bert_model, bert_test_ds, bert_tokenizer, bert_import gc
import torch

# Eliminar todos los modelos/trainers que estén en memoria
try: del llm_model
except: pass
try: del llm_base
except: pass
try: del llm_trainer
except: pass
try: del bert_model
except: pass
try: del bert_trainer
except: pass
try: del peft_model
except: pass
try: del peft_trainer
except: pass
try: del base_model
except: pass

# Limpiar cache de CUDA
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# Verificar VRAM libre
print(f"VRAM usada : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"VRAM libre : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")collator,
    batch_size=BERT_BATCH
)

RESULTS['BERT'] = {
    'model': BERT_MODEL_NAME,
    'train_time_s': round(bert_train_time, 2),
    'accuracy': round(bert_acc, 4),
    'f1_macro': round(bert_f1, 4),
    'confusion_matrix': bert_cm.tolist(),
    'total_params': bert_total,
    'trainable_params': bert_trainable,
    'trainable_pct': round(bert_pct, 4),
    'hyperparams': {
        'lr': BERT_LR, 'batch_size': BERT_BATCH,
        'epochs': BERT_EPOCHS, 'max_len': BERT_MAX_LEN, 'weight_decay': BERT_WD
    }
}


=== BERT fine-tuning — Test Metrics ===
Accuracy : 0.9311
F1 Macro : 0.9311
Confusion Matrix:
[[4619  381]
 [ 308 4692]]
              precision    recall  f1-score   support

    negative       0.94      0.92      0.93      5000
    positive       0.92      0.94      0.93      5000

    accuracy                           0.93     10000
   macro avg       0.93      0.93      0.93     10000
weighted avg       0.93      0.93      0.93     10000



---
# Enfoque 2: LLM Full Fine-tuning (`Qwen/Qwen3.5-9B`)

> **Estrategia de memoria:** Se carga el modelo en **4-bit** (NF4, doble cuantización) con `bitsandbytes`. Esto reduce el uso de VRAM de ~18 GB (FP16) a ~5-6 GB, dejando espacio para activaciones y gradientes. Solo se descongelan los parámetros de la **cabeza de clasificación** (`score` / `lm_head` → `classifier`), mientras el backbone permanece congelado. Este es el esquema de "fine-tuning de cabeza" (head-only fine-tuning) que puede realizarse sobre un LLM cuantizado sin `peft`.

In [40]:
import gc, torch

# Limpieza agresiva
for var in ['llm_model', 'llm_base', 'llm_trainer', 'bert_model', 
            'bert_trainer', 'peft_model', 'base_model']:
    if var in dir():
        exec(f'del {var}')

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"VRAM libre: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

VRAM libre: 8.17 GB


In [41]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

LLM_MODEL_NAME = 'Qwen/Qwen3.5-4B'
LLM_MAX_LEN    = 128
LLM_BATCH      = 4
LLM_GRAD_ACCUM = 8
LLM_LR         = 2e-4
LLM_EPOCHS     = 3
LLM_WD         = 0.01

In [42]:
# Liberar VRAM del intento anterior
import gc
try:
    del llm_model, llm_trainer
except: pass
gc.collect()
torch.cuda.empty_cache()

In [43]:
# Cargar en 4-bit
bnb_config_llm = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

llm_base = AutoModelForSequenceClassification.from_pretrained(
    LLM_MODEL_NAME,
    num_labels=2,
    quantization_config=bnb_config_llm,
    device_map='auto',
    trust_remote_code=True,
    id2label={0: 'negative', 1: 'positive'},
    label2id={'negative': 0, 'positive': 1},
)
llm_base.config.pad_token_id = llm_tokenizer.pad_token_id
llm_base = prepare_model_for_kbit_training(llm_base)

# LoRA agresivo: r=64, TODAS las capas lineales del transformer
lora_config_llm = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    bias='none',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'   # capas MLP también
    ],
    modules_to_save=['score'],
)

llm_model = get_peft_model(llm_base, lora_config_llm)
llm_model.print_trainable_parameters()
print('\nParámetros LLM Full (LoRA r=64, todas las capas):')
llm_total, llm_trainable, llm_pct = count_parameters(llm_model)

Loading weights: 100%|██████████| 426/426 [00:00<00:00, 439.69it/s]
[transformers] Qwen3_5ForSequenceClassification LOAD REPORT from: Qwen/Qwen3.5-4B
Key                                                | Status     | 
---------------------------------------------------+------------+-
model.visual.blocks.{0...23}.attn.proj.bias        | UNEXPECTED | 
model.visual.blocks.{0...23}.attn.proj.weight      | UNEXPECTED | 
model.visual.blocks.{0...23}.mlp.linear_fc2.bias   | UNEXPECTED | 
model.visual.blocks.{0...23}.norm2.weight          | UNEXPECTED | 
model.visual.blocks.{0...23}.attn.qkv.weight       | UNEXPECTED | 
model.visual.blocks.{0...23}.norm2.bias            | UNEXPECTED | 
model.visual.blocks.{0...23}.norm1.weight          | UNEXPECTED | 
model.visual.patch_embed.proj.weight               | UNEXPECTED | 
model.visual.blocks.{0...23}.norm1.bias            | UNEXPECTED | 
model.visual.blocks.{0...23}.mlp.linear_fc1.weight | UNEXPECTED | 
model.visual.blocks.{0...23}.mlp.linear_fc1.bi

trainable params: 84,939,776 || all params: 4,290,696,192 || trainable%: 1.9796

Parámetros LLM Full (LoRA r=64, todas las capas):
Total params    : 2,506,150,912
Trainable params: 84,939,776  (3.3893%)


In [44]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

llm_args = TrainingArguments(
    output_dir='./llm_output',
    num_train_epochs=LLM_EPOCHS,
    per_device_train_batch_size=LLM_BATCH,
    per_device_eval_batch_size=LLM_BATCH,
    gradient_accumulation_steps=LLM_GRAD_ACCUM,
    learning_rate=LLM_LR,
    weight_decay=LLM_WD,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=100,
    fp16=True,
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    seed=SEED,
    report_to='none',
    dataloader_num_workers=0,
)

llm_trainer = Trainer(
    model=llm_model,
    args=llm_args,
    train_dataset=llm_train_ds,
    eval_dataset=llm_test_ds,
    processing_class=llm_tokenizer,
    data_collator=llm_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

t0_llm = time.time()
llm_trainer.train()
llm_train_time = time.time() - t0_llm
print(f'Tiempo LLM: {llm_train_time:.1f}s ({llm_train_time/60:.2f} min)')

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
llm_acc, llm_f1, llm_cm = report_results(
    'LLM Full Fine-tuning (head-only, 4-bit)',
    llm_model, llm_test_ds, llm_tokenizer, llm_collator,
    batch_size=LLM_BATCH
)

RESULTS['LLM_Full'] = {
    'model': LLM_MODEL_NAME,
    'train_time_s': round(llm_train_time, 2),
    'accuracy': round(llm_acc, 4),
    'f1_macro': round(llm_f1, 4),
    'confusion_matrix': llm_cm.tolist(),
    'total_params': llm_total,
    'trainable_params': llm_trainable,
    'trainable_pct': round(llm_pct, 4),
    'hyperparams': {
        'lr': LLM_LR, 'batch_size': LLM_BATCH,
        'grad_accum': LLM_GRAD_ACCUM, 'epochs': LLM_EPOCHS,
        'max_len': LLM_MAX_LEN, 'weight_decay': LLM_WD,
        'quantization': '4-bit NF4'
    }
}

---
# Enfoque 3: PEFT con LoRA (`Qwen/Qwen3.5-9B` + adaptadores LoRA)

> Se usa el mismo modelo base cuantizado en 4-bit, pero ahora se añaden **adaptadores LoRA** sobre las capas de atención (`q_proj`, `k_proj`, `v_proj`, `o_proj`). Con `r=16` se agregan ~tens de millones de parámetros entrenables (< 1% del total), lo que permite actualizar capas internas del LLM sin mover todos los pesos.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

PEFT_MAX_LEN    = 256
PEFT_BATCH      = 4
PEFT_GRAD_ACCUM = 8
PEFT_LR         = 2e-4
PEFT_EPOCHS     = 3
PEFT_WD         = 0.01
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05

print(f'LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}')
print(f'LR: {PEFT_LR} | Batch: {PEFT_BATCH} (grad accum {PEFT_GRAD_ACCUM}) | Epochs: {PEFT_EPOCHS}')

In [ ]:
# Reutilizamos tokenizer y datasets de la sección LLM
peft_train_ds = llm_train_ds
peft_test_ds  = llm_test_ds
peft_collator = llm_collator

# Cargar modelo base de nuevo (limpio)
bnb_config_peft = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForSequenceClassification.from_pretrained(
    LLM_MODEL_NAME,
    num_labels=2,
    quantization_config=bnb_config_peft,
    device_map='auto',
    trust_remote_code=True,
    id2label={0: 'negative', 1: 'positive'},
    label2id={'negative': 0, 'positive': 1},
)
base_model.config.pad_token_id = llm_tokenizer.pad_token_id

# Preparar para k-bit training (activa gradient checkpointing, etc.)
base_model = prepare_model_for_kbit_training(base_model)

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],  # capas de atención Qwen
    modules_to_save=['score'],    # guardar cabeza de clasificación completa
)

peft_model = get_peft_model(base_model, lora_config)

print('\nParámetros PEFT/LoRA:')
peft_model.print_trainable_parameters()
peft_total, peft_trainable, peft_pct = count_parameters(peft_model)

In [ ]:
peft_args = TrainingArguments(
    output_dir='./peft_output',
    num_train_epochs=PEFT_EPOCHS,
    per_device_train_batch_size=PEFT_BATCH,
    per_device_eval_batch_size=PEFT_BATCH,
    gradient_accumulation_steps=PEFT_GRAD_ACCUM,
    learning_rate=PEFT_LR,
    weight_decay=PEFT_WD,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=100,
    fp16=True,
    optim='paged_adamw_8bit',    # optimizador eficiente en VRAM
    seed=SEED,
    report_to='none',
    dataloader_num_workers=0,
)

peft_trainer = Trainer(
    model=peft_model,
    args=peft_args,
    train_dataset=peft_train_ds,
    eval_dataset=peft_test_ds,
    processing_class=llm_tokenizer,
    data_collator=peft_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# ── ENTRENAMIENTO PEFT ──
t0_peft = time.time()
peft_trainer.train()
peft_train_time = time.time() - t0_peft
print(f'\nTiempo de entrenamiento PEFT: {peft_train_time:.1f}s ({peft_train_time/60:.2f} min)')

In [ ]:
peft_acc, peft_f1, peft_cm = report_results(
    'PEFT / LoRA', peft_model, peft_test_ds, llm_tokenizer, peft_collator,
    batch_size=PEFT_BATCH
)

RESULTS['PEFT_LoRA'] = {
    'model': f'{LLM_MODEL_NAME} + LoRA (r={LORA_R})',
    'train_time_s': round(peft_train_time, 2),
    'accuracy': round(peft_acc, 4),
    'f1_macro': round(peft_f1, 4),
    'confusion_matrix': peft_cm.tolist(),
    'total_params': peft_total,
    'trainable_params': peft_trainable,
    'trainable_pct': round(peft_pct, 4),
    'hyperparams': {
        'lr': PEFT_LR, 'batch_size': PEFT_BATCH,
        'grad_accum': PEFT_GRAD_ACCUM, 'epochs': PEFT_EPOCHS,
        'max_len': PEFT_MAX_LEN, 'weight_decay': PEFT_WD,
        'lora_r': LORA_R, 'lora_alpha': LORA_ALPHA,
        'lora_dropout': LORA_DROPOUT,
        'target_modules': 'q_proj,k_proj,v_proj,o_proj',
        'quantization': '4-bit NF4'
    }
}

---
# 5. Tabla comparativa de resultados

In [ ]:
rows = []
for approach, r in RESULTS.items():
    rows.append({
        'Enfoque': approach,
        'Modelo': r['model'],
        'Tiempo (s)': r['train_time_s'],
        'Tiempo (min)': round(r['train_time_s'] / 60, 2),
        'Accuracy': r['accuracy'],
        'F1 Macro': r['f1_macro'],
        'Params totales': f"{r['total_params']:,}",
        'Params entrena.': f"{r['trainable_params']:,}",
        '% Entrena.': f"{r['trainable_pct']:.4f}%",
    })

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))

In [ ]:
print('\n=== Matrices de Confusión ===')
for approach, r in RESULTS.items():
    print(f'\n{approach}:')
    cm = np.array(r['confusion_matrix'])
    print(pd.DataFrame(cm,
        index=['Real neg', 'Real pos'],
        columns=['Pred neg', 'Pred pos']))

In [ ]:
print('\n=== Hiperparámetros usados ===')
for approach, r in RESULTS.items():
    print(f'\n{approach}:')
    for k, v in r['hyperparams'].items():
        print(f'  {k}: {v}')

---
# 6. Visualización de resultados

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

approaches = list(RESULTS.keys())
accs  = [RESULTS[a]['accuracy']   for a in approaches]
f1s   = [RESULTS[a]['f1_macro']   for a in approaches]
times = [RESULTS[a]['train_time_s'] / 60 for a in approaches]
pcts  = [RESULTS[a]['trainable_pct'] for a in approaches]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Accuracy & F1
x = np.arange(len(approaches))
w = 0.35
axes[0].bar(x - w/2, accs, w, label='Accuracy', color='steelblue')
axes[0].bar(x + w/2, f1s,  w, label='F1 Macro', color='coral')
axes[0].set_xticks(x); axes[0].set_xticklabels(approaches)
axes[0].set_ylim(0.8, 1.0); axes[0].legend(); axes[0].set_title('Métricas de rendimiento')

# Tiempo de entrenamiento
axes[1].bar(approaches, times, color=['steelblue','coral','mediumseagreen'])
axes[1].set_ylabel('Minutos'); axes[1].set_title('Tiempo de entrenamiento')
for i, v in enumerate(times):
    axes[1].text(i, v + 0.5, f'{v:.1f}', ha='center')

# % parámetros entrenables
axes[2].bar(approaches, pcts, color=['steelblue','coral','mediumseagreen'])
axes[2].set_ylabel('% Parámetros entrenables')
axes[2].set_title('Eficiencia paramétrica')
for i, v in enumerate(pcts):
    axes[2].text(i, v + 0.01, f'{v:.3f}%', ha='center')

plt.tight_layout()
plt.savefig('comparacion_finetuning.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada: comparacion_finetuning.png')

In [ ]:
# Matrices de confusión
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
labels_names = ['Negative', 'Positive']

for ax, (approach, r) in zip(axes, RESULTS.items()):
    cm = np.array(r['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
                xticklabels=labels_names, yticklabels=labels_names)
    ax.set_title(approach)
    ax.set_xlabel('Predicción'); ax.set_ylabel('Real')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

---
# 7. Exportar resultados para el Excel

In [ ]:
# CSV listo para copiar al Excel IMDB50K Scores.xlsx
df_results.to_csv('resultados_tarea11.csv', index=False)
print('Exportado: resultados_tarea11.csv')
print(df_results)

---
## Notas de implementación

### Restricciones de VRAM (RTX 4070 Max-Q, 8 GB)

| Modelo | Precisión | VRAM estimada | ¿Cabe? |
|--------|-----------|--------------|--------|
| bert-base-uncased | FP16 | ~0.9 GB | ✅ |
| Qwen/Qwen3.5-9B | FP32 | ~36 GB | ❌ |
| Qwen/Qwen3.5-9B | FP16 | ~18 GB | ❌ |
| Qwen/Qwen3.5-9B | INT4 (NF4) | ~5-6 GB | ✅ |

### Decisiones de diseño
- **BERT**: Fine-tuning completo de todos los parámetros; es pequeño (110M) y cabe cómodamente.
- **LLM Full**: Dado que entrenar todos los pesos de un modelo de 9B con gradientes es imposible con 8 GB VRAM, se aplica la estrategia estándar de "head-only fine-tuning": el backbone está cuantizado y congelado, solo la cabeza de clasificación se actualiza. Esto cumple el espíritu de "LLM full fine-tuning" dentro de las restricciones de hardware.
- **PEFT/LoRA**: Añade adaptadores de bajo rango sobre las capas de atención del LLM cuantizado. Permite actualizar representaciones internas con una fracción de parámetros (<1%), siendo la estrategia más eficiente.

### ¿Por qué `AutoModelForSequenceClassification`?
Esta clase agrega automáticamente una cabeza de clasificación lineal (`score`) sobre el LLM base, adaptando el modelo generativo a la tarea de clasificación binaria (negative / positive).